# 02 - Partición temporal y baseline

Antes de entrenar hay que decidir **cómo evaluar**. Con datos de series de tiempo el split aleatorio
(`train_test_split`) que usamos en clase es una trampa: mezcla pasado y futuro. Acá lo mostramos.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RAIZ = Path.cwd().parent          # el notebook corre desde notebooks/
sys.path.append(str(RAIZ / "backend"))  # para importar app.dataset y app.modelo
DATA = RAIZ / "data"

plt.rcParams["figure.figsize"] = (8, 4)

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from app.modelo import FEATURES, TARGET, GAP_CORTES, dividir_temporal, predecir_baseline, metricas

dataset = pd.read_csv(DATA / "dataset_pares.csv", parse_dates=["fecha_corte"])
print(dataset.shape)

(48285, 11)


## 1. Split temporal con gap

`dividir_temporal` manda las **últimas fechas** a test y descarta unas fechas en el medio (*gap*).

¿Por qué el gap? El target de una fecha de corte mira 60 ruedas hacia adelante. Las fechas de corte están
separadas 20 ruedas, así que el target del último corte de train llega hasta 3 cortes después. Sin gap,
ese target se solaparía con el primer corte de test.

In [3]:
train, test = dividir_temporal(dataset)
print("GAP_CORTES:", GAP_CORTES)
for nombre, d in [("train", train), ("test", test)]:
    print(f"{nombre:5s} {len(d):6d} filas | {d.fecha_corte.nunique():3d} cortes | {d.fecha_corte.min().date()} a {d.fecha_corte.max().date()}")

fechas = np.sort(dataset.fecha_corte.unique())
descartadas = [f for f in fechas if f not in set(train.fecha_corte) | set(test.fecha_corte)]
print("fechas descartadas (gap):", [str(pd.Timestamp(f).date()) for f in descartadas])

GAP_CORTES: 3
train  32625 filas |  75 cortes | 2014-12-30 a 2020-11-13
test   14355 filas |  33 cortes | 2021-03-12 a 2023-09-27


fechas descartadas (gap): ['2020-12-14', '2021-01-13', '2021-02-11']


### Comprobación: el gap evita que train y test compartan días

El target de cada fila abarca las 60 ruedas siguientes a su fecha de corte. Para el **último corte de train** y
el **primer corte de test** calculamos cuántas ruedas de esas ventanas se pisan. Con el gap tiene que dar 0;
sin gap (o con uno más chico) da más de 0, y eso es la fuga de información.

In [4]:
from app.dataset import HORIZONTE

precios = pd.read_csv(DATA / "precios.csv", index_col="fecha", parse_dates=True)
ruedas = precios.pct_change().dropna().index   # mismo calendario que usa el dataset

def solapamiento(gap):
    """Ruedas compartidas entre el target del último corte de train y el del primer corte de test."""
    tr, te = dividir_temporal(dataset, gap_cortes=gap)
    i = ruedas.get_loc(tr.fecha_corte.max())    # último corte de train
    j = ruedas.get_loc(te.fecha_corte.min())    # primer corte de test
    return max(0, (i + HORIZONTE) - (j + 1) + 1)  # target train: i+1..i+60 | target test: j+1..j+60

# Con el gap real: fechas concretas
i = ruedas.get_loc(train.fecha_corte.max())
print("último corte de train:      ", ruedas[i].date())
print("su target llega hasta:      ", ruedas[i + HORIZONTE].date())
print("primer corte de test:       ", test.fecha_corte.min().date())
print("ruedas entre ambos:         ", ruedas.get_loc(test.fecha_corte.min()) - (i + HORIZONTE))

pd.DataFrame({
    "gap (cortes)": range(0, GAP_CORTES + 1),
    "ruedas de target compartidas": [solapamiento(g) for g in range(0, GAP_CORTES + 1)],
}).set_index("gap (cortes)")

último corte de train:       2020-11-13
su target llega hasta:       2021-02-11
primer corte de test:        2021-03-12
ruedas entre ambos:          20


,ruedas de target compartidas
gap (cortes),
0,40
1,20
2,0
3,0


In [5]:
assert solapamiento(GAP_CORTES) == 0, "con el gap elegido no debe haber solapamiento"
assert solapamiento(0) > 0, "sin gap sí debería haber solapamiento"
print("OK: con GAP_CORTES =", GAP_CORTES, "no se comparte ninguna rueda de target; sin gap se comparten", solapamiento(0))

OK: con GAP_CORTES = 3 no se comparte ninguna rueda de target; sin gap se comparten 40


## 2. Por qué no un split aleatorio

Entrenamos el mismo modelo (kNN) de dos formas y lo evaluamos en test:

- **aleatorio:** mezcla filas de todas las fechas. Para una fila de test casi siempre hay filas de train
  de la **misma fecha** (otros pares, mismo mercado) y de fechas vecinas.
- **temporal:** entrena en el pasado, evalúa en el futuro (lo que pasaría en la vida real).

In [6]:
def knn():
    return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=15))

tr_a, te_a = train_test_split(dataset, test_size=0.3, random_state=0)
m_aleatorio = knn().fit(tr_a[FEATURES], tr_a[TARGET])
m_temporal = knn().fit(train[FEATURES], train[TARGET])

pd.DataFrame({
    "split aleatorio": metricas(te_a[TARGET], m_aleatorio.predict(te_a[FEATURES])),
    "split temporal": metricas(test[TARGET], m_temporal.predict(test[FEATURES])),
}).round(3)

,split aleatorio,split temporal
mae,0.158,0.187
rmse,0.200,0.232
r2,0.349,-0.019


El split aleatorio da un error más bajo y un R² más alto: es **optimista** porque el modelo vio
filas casi idénticas a las de test. Esa es la cifra que *no* se puede esperar en producción. Por eso el
informe usa solo el split temporal.

## 3. Baseline

Un modelo solo vale si le gana a algo simple. El baseline es **"la correlación futura será igual a la
de los últimos 90 días"** (`corr_90d`): no entrena nada y es lo que haría cualquiera sin ML.
Comparamos también con predecir siempre la media del train.

In [7]:
pred_baseline = predecir_baseline(test)
pred_media = np.full(len(test), train[TARGET].mean())

pd.DataFrame({
    "baseline (corr_90d)": metricas(test[TARGET], pred_baseline),
    "siempre la media de train": metricas(test[TARGET], pred_media),
}).round(3)

,baseline (corr_90d),siempre la media de train
mae,0.161,0.186
rmse,0.204,0.230
r2,0.211,-0.001


- **MAE:** error absoluto promedio, en unidades de correlación (fácil de interpretar).
- **RMSE:** penaliza más los errores grandes.
- **R²:** qué fracción de la varianza explica; 0 = no mejor que la media, negativo = peor que la media.

Cualquier modelo del próximo notebook tiene que superar al baseline en test para justificarse.